## Kode buat ngasih target ec ke dataset sintetisnya

### Rangkuman Pembelaan: Kenapa Target EC Naik Turun?

Pendekatan ini menggunakan konsep **VPD (Vapor Pressure Deficit)** dan **Tekanan Osmotik**. 

**Prinsip dasarnya:**

1. **Saat tanaman BANYAK minum (suhu panas / udara kering)**
   Target EC diturunkan agar larutan lebih ringan disedot oleh akar. Hal ini bertujuan untuk mencegah penumpukan garam berlebih yang bisa menyebabkan keracunan atau *nutrient burn*.
   
2. **Saat tanaman SEDIKIT minum (suhu dingin / udara pengap)**
   Target EC dinaikkan agar nutrisi di dalam air menjadi lebih pekat. Hal ini memastikan kebutuhan gizi harian tanaman tetap terpenuhi secara maksimal, meskipun volume air yang masuk ke akar hanya sedikit.

In [1]:
import pandas as pd

df = pd.read_csv('dataset/dataset_paprika.csv')

df['EC_Tangki'] = pd.to_numeric(df['EC_Tangki'], errors='coerce')
df['Suhu_Udara'] = pd.to_numeric(df['Suhu_Udara'], errors='coerce')
df['Kelembapan_Udara'] = pd.to_numeric(df['Kelembapan_Udara'], errors='coerce')

def modifier_suhu(suhu):
    if suhu > 28.0:
        # Suhu Panas: Transpirasi tinggi, tanaman nyedot banyak air buat pendinginan. 
        # EC dikurangi (-0.2) untuk mencegah penumpukan garam/nutrient burn.
        return -0.2
    elif suhu < 22.0:
        # Suhu Dingin: Metabolisme turun, tanaman minum sedikit. 
        # EC ditambah (+0.1) agar larutan lebih pekat dan nutrisi tetap cukup.
        return 0.1
    else:
        # Suhu ideal, tidak ada modifikasi.
        return 0.0

def modifier_kelembapan(rh):
    if rh < 60.0:
        # RH Rendah (Kering): Daun ditarik udara, transpirasi tinggi. 
        # EC dikurangi (-0.1) biar akar nggak berat ngelawan tekanan osmotik pekat.
        return -0.1
    elif rh > 80.0:
        # RH Tinggi (Pengap): Transpirasi mampet, air susah menguap. 
        # EC ditambah (+0.1) agar sedikit air yang berhasil disedot bawa nutrisi maksimal.
        return 0.1
    else:
        # Kelembapan ideal, tidak ada modifikasi.
        return 0.0

mod_suhu = df['Suhu_Udara'].apply(modifier_suhu)
mod_lembab = df['Kelembapan_Udara'].apply(modifier_kelembapan)

base_ec = df['EC_Tangki']

# Kalkulasi akhir target EC dengan menjumlahkan base EC dan modifier dari iklim
df['target_ec'] = (base_ec + mod_suhu + mod_lembab).round(2)

df.to_csv('dataset/dataset_paprika_ec.csv', index=False)

print("Kolom target_ec berhasil dibuat. File tersimpan.")

Kolom target_ec berhasil dibuat. File tersimpan.
